# LoRA Demo: Efficient Fine-Tuning of a Large Language Model

In this notebook, we will demonstrate how to fine-tune a pretrained large language model using **LoRA (Low-Rank Adaptation)**.

This demo is meant to help build intuition for:

- loading a pretrained LLM
- using **8-bit quantization** to reduce memory usage
- adding **LoRA adapters** to selected transformer layers
- training only a small number of parameters
- running inference after fine-tuning


## Why LoRA?

Large language models are very expensive to fine-tune if we update all parameters.  
That requires a lot of:

- GPU memory
- training time
- storage

LoRA solves this by **freezing the original pretrained model** and learning only a **small low-rank update** for selected layers.

This makes LoRA a popular **Parameter-Efficient Fine-Tuning (PEFT)** method.



This notebook focuses on the practical mechanics of LoRA:

1. Install libraries and log in
2. Load a pretrained model using quantization
3. Add LoRA adapters
4. Prepare a small dataset
5. Fine-tune only the LoRA parameters
6. Merge the adapters back into the model
7. Test the fine-tuned model on custom prompts


In [ ]:
!pip install -q bitsandbytes datasets accelerate loralib
# bitsandbytes: Provides 8-bit and 4-bit quantization for memory-efficient training
# accelerate: Optimizes training performance across different hardware configurations
# loralib: Core library implementing LoRA (Low-Rank Adaptation) technique
# !pip install -q git+https://github.com/huggingface/transformers.git@main git+https://github.com/huggingface/peft.git
!pip install "transformers[torch]" peft
# transformers: Latest version with LoRA support and bug fixes
# peft: Parameter-Efficient Fine-Tuning library that implements LoRA and other PEFT methods

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 17.1 MB/s eta 0:00:00


## Step 1: Install required libraries

We first install the libraries needed for:

- model loading
- quantization
- LoRA / PEFT
- dataset handling
- training utilities

### Important libraries
- **bitsandbytes**: enables 8-bit or 4-bit quantization for memory-efficient model loading
- **transformers**: used to load and train Hugging Face language models
- **peft**: implements parameter-efficient fine-tuning methods such as LoRA
- **datasets**: used to load and preprocess datasets
- **accelerate**: helps optimize training across available hardware

In [ ]:
from huggingface_hub import login

# You can put your Hugging Face token in the environment variable HF_TOKEN
# Otherwise, it will pop up with a authorization link.
login()

## Step 3: Load the pretrained model and apply LoRA

In this step, we do three important things:

1. Load the pretrained base model  
2. Use **8-bit quantization** to reduce memory usage  
3. Add **LoRA adapters** to selected attention layers  


### Why use quantization?

The base model in this demo is **Mistral-7B**, which is a large language model.  
Loading such a model in full precision would require much more memory.

So we use **8-bit quantization**, which stores weights in a compressed form and reduces memory usage significantly.



### Why use LoRA?

Instead of updating all model weights, LoRA keeps the base model **frozen** and learns a small trainable update for selected layers.

In this demo, LoRA is applied to:

- `q_proj`
- `v_proj`

These are the **query** and **value** projection layers inside the transformer attention blocks.

This is a common choice because attention layers play a major role in how transformers process context.



### Why do we print trainable parameters?

This is one of the most important checks in a LoRA notebook.

If LoRA is applied correctly:

- the **total number of parameters** stays very large
- the **trainable parameters** become only a small fraction of the total

That is the practical evidence that we are doing **parameter-efficient fine-tuning**, not full fine-tuning.

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig #(provides tools for LLMs on hardware with limited memory with help of qunatization)
from peft import LoraConfig, PeftModel, get_peft_model

In [ ]:
# Define base model (Mistral 7B)
base_model_name = "mistralai/Mistral-7B-v0.1"

In [ ]:
# Enable 8-bit quantization for efficient training
bnb_config = BitsAndBytesConfig(
    load_in_8bit=True,  # Switch to load_in_4bit=True if needed, Reduces model memory usage by ~50% by using 8-bit instead of 32-bit precision
    llm_int8_enable_fp32_cpu_offload=True  # Allow CPU offloading to save GPU memory, Moves some computations to CPU when GPU memory is full
)

In [ ]:
# Load base model with quantization
base_model = AutoModelForCausalLM.from_pretrained(
    base_model_name,
    quantization_config=bnb_config, #Applies the 8-bit quantization we configured above
    device_map="auto",
    dtype=torch.float16 #base data type for model weights and computations
)

config.json:   0%|          | 0.00/571 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/25.1k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

In [ ]:
# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(base_model_name) # tokenizer converts text to numbers (tokens), must match the base model to ensure compatibility

tokenizer_config.json:   0%|          | 0.00/996 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.80M [00:00<?, ?B/s]

tokenizer.model: reconstructing file:   0%|          |  0.00B /  493kB            

tokenizer.model: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

In [ ]:
# Define LoRA Configuration
lora_config = LoraConfig(
    r=16,                                    # Rank of the low-rank matrices (higher = more parameters, more capacity)
    lora_alpha=32,                           # Scaling factor for LoRA weights (controls adaptation strength), Scales LoRA updates by 32/r = 2, making adaptations more prominent
    target_modules=["q_proj", "v_proj"],     # attention modules, Only adapts Query and Value projection matrices in attention
    lora_dropout=0.05,                       # Dropout rate for LoRA layers (prevents overfitting)
    bias="none",                             # Don't adapt bias terms (keeps original model biases)
    task_type="CAUSAL_LM"
)

In [ ]:
# Apply LoRA fine-tuning
lora_model = get_peft_model(base_model, lora_config)

In [ ]:
# Print trainable parameters
def print_trainable_parameters(model):
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total_params = sum(p.numel() for p in model.parameters())
    print(f"Trainable Params: {trainable_params} / Total Params: {total_params} ({100 * trainable_params / total_params:.2f}%)")


In [ ]:
print_trainable_parameters(lora_model)

print("LoRA model setup complete!")

Trainable Params: 6815744 / Total Params: 7248547840 (0.09%)
LoRA model setup complete!


## Understanding model precision during loading, training, and inference

A useful way to understand this demo is to track what happens to the model precision at different stages.

### Model Loading
**fp32 -> fp16 -> 8-bit quantized (for storage)**

The original pretrained model may start in higher precision, but for efficient loading it is stored in **8-bit quantized form** to save memory.

### Training
**8-bit -> fp16 (dequantize) -> fp16 computations -> fp16 gradients -> fp16 updates**

During training, the quantized weights are dequantized when needed for computation.  
The forward pass and backward pass happen in floating point precision, while only the LoRA parameters are updated.

### Inference
**8-bit -> fp16 (dequantize) -> fp16 computations**

During inference, the same idea applies: the weights may be stored compactly, but computations are still performed using floating point operations.

### Key takeaway
- **Quantization** reduces memory needed to store the base model
- **LoRA** reduces the number of trainable parameters

These two techniques solve different parts of the efficiency problem.

## Step 4: Load and preprocess the dataset

Now we prepare a small dataset for fine-tuning.

In this demo, we use the **English Quotes** dataset.

Each example contains:
- a quote
- associated tags

We then combine them into a single formatted sequence:

`quote ->: tags`

### Why format the data this way?

This model is trained as a **causal language model**, which means it learns to predict the next token in a sequence.

So by formatting the data like this, we teach the model a continuation pattern:

- given the quote,
- generate the tag-related continuation that follows it.

This is a simple and practical way to create training examples for language-model fine-tuning.

In [ ]:
from datasets import load_dataset

In [ ]:
# Load dataset (English Quotes)
data = load_dataset("Abirate/english_quotes") #he dataset contains quotes with associated tags (categories)
#The model will learn to generate text in the style of quotes and understand tag associations

README.md:   0%|          | 0.00/5.55k [00:00<?, ?B/s]

quotes.jsonl: reconstructing file:   0%|          |  0.00B /  647kB            

quotes.jsonl: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/2508 [00:00<?, ? examples/s]

In [ ]:
# Merge columns to create a formatted dataset
def merge_columns(example):
    example['prediction'] = example["quote"] + " ->: " + str(example['tags']) # creating input-output pairs for fine tuning
    return example
#creates a training format where:
# - Input: A quote
# - Output: The same quote followed by " ->: " and the associated tags
# Example: "Life is beautiful" ->: ['life', 'beauty', 'inspiration']
# This teaches the model to understand the relationship between quotes and their categories

In [ ]:
# Apply column merge
data = data.map(merge_columns)

Map:   0%|          | 0/2508 [00:00<?, ? examples/s]

In [ ]:
# The tokenizer converts text like "Life is beautiful ->: ['life', 'beauty']" into token IDs
data = data.map(lambda samples: tokenizer(samples['prediction']), batched=True)
print("Dataset preprocessed successfully!")

Map:   0%|          | 0/2508 [00:00<?, ? examples/s]

Dataset preprocessed successfully!


## Step 5: Fine-tune the model

Now we train the model using the processed dataset.

### Important idea

Even though the training loop looks like ordinary fine-tuning, the process is different from full fine-tuning:

- the **base model weights are frozen**
- only the **LoRA adapter parameters** are trained

This is the central idea behind LoRA.


### Why is the batch size small?

The model is large, so we keep:

- batch size small
- training short
- fp16 enabled

This makes the demo practical to run on limited hardware.


### Why disable cache during training?

Caching is useful during inference because it speeds up token generation.  
But during training it is not needed and can waste memory.

So we disable it to make training more memory-efficient.


### What gets saved?

After training, we save only the **LoRA adapters**, not the full base model.

That is another major advantage of LoRA:
- smaller storage
- reusable task-specific adapters
- no need to save a full separate copy of the large model

In [ ]:
import transformers
from transformers import TrainingArguments, Trainer, DataCollatorForLanguageModeling

In [ ]:
# Fix: Ensure tokenizer has a pad_token, this make sures that all sequences of same length during training
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token  # Use EOS token for padding if available
    if tokenizer.pad_token is None:
        tokenizer.add_special_tokens({'pad_token': '[PAD]'})  # Add a custom PAD token

In [ ]:
# Define training arguments
training_args = TrainingArguments(
    per_device_train_batch_size=1,    # Process 1 example at a time per GPU (memory constraint)
    gradient_accumulation_steps=1,    # Accumulate gradients over 1 step (no accumulation)
    warmup_steps=10,                  # Gradually increase learning rate over first 10 steps
    max_steps=50,                     # Short run for testing (only 50 training steps)
    learning_rate=2e-4,               # Learning rate for LoRA parameters (2e-4 is good for LoRA)
    fp16=True,                        # Use 16-bit precision to save memory
    logging_steps=1,                  # Log training progress every step
    output_dir="outputs"              # Directory to save the trained model
)

In [ ]:
# Define trainer
trainer = Trainer(
    model=lora_model, #The lora model we created above, only LoRA parameters will be trained
    train_dataset=data['train'],
    args=training_args,
    data_collator=DataCollatorForLanguageModeling(tokenizer, mlm=False)# Handles batching and padding, Uses causal language modeling (predict next token) instead of MLM
)

In [ ]:
# Disable cache during training
# Cache is useful for inference but not needed during training; to save memory we are setting it to false
lora_model.config.use_cache = False

In [ ]:
# Start training/Fine-tunign process
trainer.train()
# 1. Load batches from the dataset
# 2. Forward pass through the model
# 3. Calculate loss (how well the model predicts the next token)
# 4. Backward pass to update LoRA parameters
# 5. Repeat for max_steps (50 steps in this demo)


Step,Training Loss
1,2.394165
2,2.406800
3,3.454350
4,3.603763
5,2.605125
6,1.805955
7,1.644808
8,2.333603
9,1.889711
10,2.111047


TrainOutput(global_step=50, training_loss=1.4414938819408416, metrics={'train_runtime': 47.1841, 'train_samples_per_second': 1.06, 'train_steps_per_second': 1.06, 'total_flos': 138235615764480.0, 'train_loss': 1.4414938819408416, 'epoch': 0.019936204146730464})

In [ ]:
# Save the fine-tuned model
lora_model.save_pretrained("outputs") # This saves the trained LoRA adapters (lora parameters) to the "outputs" directory, not the base model

print("Training complete! Model saved.")

Training complete! Model saved.


## Step 6: Load the trained adapter and merge it for inference

After training, the LoRA adapter is stored separately from the base model.

At inference time, we load the adapter and combine it with the pretrained model.

### Why merge the adapter?

Merging incorporates the learned LoRA updates directly into the base model weights, which makes inference cleaner and more efficient.

So conceptually:

- during training, the adapter stays separate
- during inference, it can be merged into one usable model

In [ ]:
# Load the fine-tuned LoRA model
fine_tuned_model = PeftModel.from_pretrained(base_model, "outputs") # combines the base model with the trained LoRA adapters, Now model has the base capabilities plus the fine-tuned adaptations

# Merge LoRA adapter into base model for efficient inference
fine_tuned_model = fine_tuned_model.merge_and_unload() #combines the LoRA adapters directly into the base model weights to create a single, unified model

print("LoRA Model Loaded Successfully!")

/usr/local/lib/python3.13/dist-packages/peft/tuners/tuners_utils.py:305: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/peft/tuners/lora/bnb.py:99: UserWarning: Merge lora module to 8-bit linear may get different generations due to rounding errors.
  warnings.warn(


LoRA Model Loaded Successfully!


## Step 7: Test the fine-tuned model on custom prompts

Now we run inference using custom inputs.

The prompt format matches the training format:

`quote ->:`

The model then tries to generate the continuation.

This helps us check whether the fine-tuned model has learned the pattern introduced during training.

### What to observe
When looking at the outputs, ask:

- Does the model follow the expected format?
- Does it generate tag-like or continuation-like text?
- Does it appear to have adapted to the fine-tuning pattern?

This is the practical way to inspect what the LoRA fine-tuning achieved.

In [ ]:
# Define a prompt for inference
prompt = "“Hurray! exam got cancelled today becuase of AWS operational issuse” ->: "

In [ ]:
# Tokenize input
batch = tokenizer(prompt, return_tensors="pt")

In [ ]:
# Move input tensors to the correct device
batch = {k: v.to(base_model.device) for k, v in batch.items()}

In [ ]:
# Generate response
with torch.no_grad():
    output_tokens = fine_tuned_model.generate(
        **batch,
        max_new_tokens=30,
        temperature=0.7,  # Adjust creativity
        top_p=0.9,  # Nucleus sampling
        repetition_penalty=1.2,  # Prevent repetition
        do_sample=True  # Enable sampling
    )


In [ ]:
# Decode and print the response
generated_text = tokenizer.decode(output_tokens[0], skip_special_tokens=True)
print("\nGenerated Response:\n", generated_text)


Generated Response:
 “Hurray! exam got cancelled today becuase of AWS operational issuse” ->:  ['aws', 'exam']

['solution-to-problems', 'thoughts'].include?((/^[


In [ ]:
# Define a prompt for inference
prompt = "“If you want the best AI engineer, then hire me” ->: "

# Tokenize input
batch = tokenizer(prompt, return_tensors="pt")

# Move input tensors to the correct, same device
batch = {k: v.to(base_model.device) for k, v in batch.items()}

# Generate response
with torch.no_grad():
    output_tokens = fine_tuned_model.generate(
        **batch,
        max_new_tokens=50,
        temperature=0.7,  # Adjust creativity
        top_p=0.9,  # Nucleus sampling
        repetition_penalty=1.2,  # Prevent repetition
        do_sample=True  # Enable sampling instead of greedy decoding
    )

# Decode and print the response
generated_text = tokenizer.decode(output_tokens[0], skip_special_tokens=True)
print("\nGenerated Response:\n", generated_text)



Generated Response:
 “If you want the best AI engineer, then hire me” ->:  ['ai', 'artificial-intelligence']
['interviewing', 'ruby'] = > ['developer', 'engineering', 'hiring', 'programming', 'rails', 'software-development'] =>


## Final takeaway

This notebook demonstrates the practical intuition behind **LoRA-based fine-tuning**.

### Main ideas from this demo
- Large language models are expensive to fine-tune fully
- **Quantization** reduces the memory needed to load the base model
- **LoRA** reduces the number of trainable parameters
- The pretrained model remains mostly frozen
- Only a small set of adapter parameters are trained
- After training, the adapter can be merged back into the model for inference

### Big picture
LoRA is powerful because it makes fine-tuning much more practical without retraining the entire network.

That is why it is one of the most widely used methods in **parameter-efficient fine-tuning (PEFT)**.